# 类型收窄与控制流分析

学习目标：能通过真实条件检查缩小类型范围，并用联合、守卫和穷尽检查可靠地处理分支。

前置知识：JavaScript 条件分支、相等判断、异常和函数；TypeScript 联合、字面量和 unknown。

适用版本：TypeScript 7.0.2、Node.js 24.11.0；ES 模块，开启 strict。

环境准备：[环境配置与运行](README.md)。

工作目录：content/编程语言/typescript。

配套脚本：位于 scripts/06-narrowing/。

1. [main.ts](scripts/06-narrowing/main.ts)：按正文顺序组织的正常示例，片段依赖同文件前文定义。
2. [type-errors.ts](scripts/06-narrowing/type-errors.ts)：与正常示例隔离的类型反例，不生成或执行 JavaScript。
3. [tsconfig.json](scripts/06-narrowing/tsconfig.json)、[tsconfig.errors.json](scripts/06-narrowing/tsconfig.errors.json)：分别明确正常与反例文件范围。
4. [lying-guard-runtime-error.ts](scripts/06-narrowing/lying-guard-runtime-error.ts)：单独说明和执行的配套边界示例。



Step 1：检查正常项目的类型。

```bash
npm run check:06
```

Step 2：生成正常项目的 JavaScript。

```bash
npm run build:06
```

Step 3：运行正常示例。

```bash
npm run run:06
```

Step 4：检查下文独立列出的类型反例。

```bash
npm run errors:06
# 预期非零退出；按反例注释逐行核对具体错误，不运行 type-errors.ts。
```

正常配置只包含上面列出的正常与独立运行示例，生成文件位于 .build/06-narrowing/。错误配置继承正常选项，改用 type-errors.ts 并开启 noEmit。

## 1 typeof、真值与空字符串

类型收窄（narrowing）是在某个控制流位置把可能类型缩小为更具体范围。typeof 检查与 JavaScript 的运行时标签对应；尤其 typeof null 为 object，因此检查对象前仍需排除 null。

真值判断会排除空字符串、0 等假值，不仅是 null 和 undefined。若空字符串是有效输入，应显式判断空值，避免把“长度为零”误判为“没有提供”。下面两个函数故意对空字符串给出不同结果。

```typescript
import assert from "node:assert/strict";
function textSize(value: string | null | undefined): number {
  if (value === null || value === undefined) return -1;
  return value.length;
}
function truthySize(value: string | null): number { return value ? value.length : -1; }
function category(value: unknown): string {
  if (typeof value === "string") return value.toUpperCase();
  if (typeof value === "object" && value !== null) return "对象";
  return "其他";
}
assert.equal(textSize(""), 0);
assert.equal(textSize(null), -1);
assert.equal(textSize(undefined), -1);
console.log(textSize(""), truthySize(""), category(null), category({})); // 0 -1 其他 对象
```

以下片段来自独立的 type-errors.ts：

```typescript
function unsafeObject(value: object | null): string {
  if (typeof value === "object") return value.toString(); // object 标签也可能是 null。
  return "其他";
}
```

## 2 相等、in 与 instanceof

当两个联合值严格相等时，分支可保留两者共有的可能类型。下面左边是 string | number，右边是 string | boolean，相等分支中两者均为 string。

in 判断对象自身或原型链上是否有给定属性；必有、可选、缺失属性对两个分支的收窄不同，可选属性对应的类型可能留在两边。它不保证属性值满足业务类型。instanceof 通常依据构造函数的原型关系收窄，不是对任意结构相似的对象执行字段验证。

```typescript
function common(left: string | number, right: string | boolean): string {
  return left === right ? left.toUpperCase() : "不同";
}
type Message = { text: string } | { code: number };
function format(message: Message): string {
  return "text" in message ? message.text : String(message.code);
}
type MaybeText = { text?: string } | { code: number };
function hasText(value: MaybeText): string {
  if ("text" in value) return value.text ?? "未设置";
  return "没有 text 属性"; // 此处仍可能是省略 text 的第一种成员。
}
function failureLabel(value: Error | string): string {
  return value instanceof Error ? value.message : value;
}
console.log(common("ts", "ts"), format({ code: 7 }), hasText({}), failureLabel(new Error("停止"))); // TS 7 没有 text 属性 停止
```

## 3 赋值与可达控制流

赋值会更新当前观察到的类型，但变量声明的可赋值范围仍然有效。声明为 string | number 后，先赋字符串可以调用字符串方法，随后仍可赋数值。

控制流分析（control flow analysis）综合分支、提前返回和赋值。若数值分支已经返回，后续代码就不可能接收到该数值分支。合流后，仍需根据所有可达路径判断类型，不应把一次收窄看成永久改变变量声明。

```typescript
let changing: string | number = "ts";
const upper = changing.toUpperCase();
changing = 4;
function pad(value: string | number): string {
  if (typeof value === "number") return value.toFixed(1);
  return value.trim(); // 数值分支已返回，这里只剩 string。
}
console.log(upper, changing.toFixed(0), pad(" 类型 "), pad(2)); // TS 4 类型 2.0
```

以下片段来自独立的 type-errors.ts：

```typescript
let declared: string | number = "x";
declared = true; // 收窄不改变声明允许的 string | number 范围。
```

## 4 可辨识联合与 never 穷尽检查

把状态与字段放在同一个联合成员中，检查状态就同时确定了可用数据。

可辨识联合（discriminated union）把状态与该状态的数据放在一起。Job 的三个成员都有 kind，但取不同字面量：done 才带 count，failed 才带 reason。检查 kind 即可选择对应成员；把所有字段写成可选属性会丢失这种关系，不能靠非空断言补回来。

所有成员都已返回后，剩余值应为 never，表示没有可能值。把剩余值赋给 never 或传给只接受 never 的函数，可以在新增分支却忘记处理时触发诊断。最后的抛错是运行时防线，不替代输入校验。

![判别字段把联合分到各自的数据分支。Job 的 kind 字段决定本分支能读取哪些关联字段。](image/illustration/06-01-discriminated-union.svg)

图示说明：箭头表示控制流进入对应分支后的类型关系，不表示运行时会创建三个对象。

阅读下面 switch 时，对照每个 case 中允许访问的字段；再看 AddedState 反例中哪条新增路径没有返回。

```typescript
type Job = { kind: "idle" } | { kind: "done"; count: number } | { kind: "failed"; reason: string };
function unreachable(value: never): never { throw new Error("未处理状态"); }
function describe(job: Job): string {
  switch (job.kind) {
    case "idle": return "等待";
    case "done": return "完成:" + job.count;
    case "failed": return "失败:" + job.reason;
    default: return unreachable(job);
  }
}
assert.equal(describe({ kind: "idle" }), "等待");
assert.equal(describe({ kind: "failed", reason: "格式" }), "失败:格式");
console.log(describe({ kind: "done", count: 0 })); // 完成:0
```

以下片段来自独立的 type-errors.ts：

```typescript
type AddedState = { kind: "done" } | { kind: "paused" };
function incomplete(value: AddedState): string {
  if (value.kind === "done") return "完成";
  const rest: never = value; // paused 尚未处理，不能赋给 never。
  return rest;
}
```

## 5 类型谓词必须与实际检查相符

类型谓词 value is User 的 value 必须对应当前形参，User 必须可赋给该形参类型。返回 true 的函数分支为调用者提供“这是 User”的承诺；编译器不会证明所有检查是否足以支撑这个承诺。

下面先接受非 null 对象或函数，再检查 name 存在且为字符串；函数也可凭字符串 name 结构性满足 User。空字符串同样合法；若业务要求非空，应该另加业务检查，不能让一个表示所有 User 的守卫错误地排除合法 User。额外成员不影响本例的结构契约。

```typescript
type User = { name: string };
function isUser(value: unknown): value is User {
  return ((typeof value === "object" && value !== null) || typeof value === "function") &&
    "name" in value && typeof value.name === "string";
}
function named() {}
const candidates: unknown[] = [{ name: "林" }, { name: "" }, null, {}, { name: 3 }, [], named];
assert.deepEqual(candidates.map(isUser), [true, true, false, false, false, false, true]);
const users: User[] = candidates.filter(isUser);
console.log(users.map((user) => user.name.length).join(",")); // 1,0,5
```

## 6 断言函数与失实守卫的后果

asserts value is User 表示函数正常返回后可按 User 使用；无效输入必须抛出异常，不能静默返回。asserts condition 则断言某个条件为真。它们描述控制流，与 as 类型断言不同：真正的运行时检查来自函数体。

本例使用 Node 的严格断言核对有效、无效与边界输入。assert.throws 检查异常的类别与消息，正常输入则在返回后直接使用 name。另有独立的失实守卫反例：它仅检查对象存在，却声称 name 一定是字符串。

```typescript
function assertUser(value: unknown): asserts value is User {
  if (!isUser(value)) throw new TypeError("需要字符串 name");
}
function assertCondition(condition: unknown): asserts condition {
  if (!condition) throw new Error("条件不满足");
}
const valid: unknown = { name: "" };
assertUser(valid);
assertCondition(valid.name.length === 0);
for (const invalid of [null, undefined, {}, { name: 1 }]) {
  assert.throws(() => assertUser(invalid), { name: "TypeError", message: "需要字符串 name" });
}
console.log(valid.name.length); // 0
```

## 7 独立运行失实守卫反例

下面是配套 lying-guard-runtime-error.ts 的完整内容。这个签名可以通过检查，但读取 number 的字符串方法会失败；不要把“函数返回了布尔值”视为其类型谓词正确的证据。

```typescript
export {};
function lyingGuard(value: unknown): value is { name: string } {
  return typeof value === "object" && value !== null;
}
const input: unknown = { name: 3 };
if (lyingGuard(input)) input.name.toUpperCase(); // 运行时 TypeError：守卫没有检查 name 的类型。
```

Step 1：在已执行 build:06 后单独运行反例。

```bash
node .build/06-narrowing/lying-guard-runtime-error.js
# 预期退出码 1；TypeError，消息包含 input.name.toUpperCase is not a function。
```

## 本章小结

收窄来自具体可达路径和真实条件。空字符串与空值要分别考虑；可辨识联合保留状态与数据的关联，never 检查遗漏，自定义守卫和断言函数仍需验证实现。

## 练习

1. 给 Job 新增 paused 成员，先确认穷尽检查失败，再加入分支；所有状态均应返回明确文本。
2. 为 User 增加可选年龄，年龄存在时要求非负有限数值，测试省略、0、负数、字符串、NaN；区分结构类型与业务规则。
3. 修复 lyingGuard，使数值 name 被拒绝，空字符串 name 被接受；运行不再发生 TypeError，不能使用 as 或 !。
4. 在 Job 中增加 kind 为 paused、带 remaining: number 的成员；先在分支图补路，再为 describe 增加处理，核对穷尽检查由失败变为通过。

## 参考与引用来源

- TypeScript 官方文档：[Narrowing：typeof 至穷尽性检查](https://www.typescriptlang.org/docs/handbook/2/narrowing.html)；[3.7：Assertion Functions](https://www.typescriptlang.org/docs/handbook/release-notes/typescript-3-7.html#assertion-functions)；[4.9：in 对未列出属性的收窄](https://www.typescriptlang.org/docs/handbook/release-notes/typescript-4-9.html#unlisted-property-narrowing-with-the-in-operator)。

- Node.js 24.11.0：[Assert：strict、deepEqual、equal、throws](https://nodejs.org/download/release/v24.11.0/docs/api/assert.html)。

- ECMA-262 第 16 版（ECMAScript 2025）：[20.2.4.2：函数 name 是字符串](https://262.ecma-international.org/16.0/#sec-function-instances-name)。

- npm 官方文档：[npm run（v11）](https://docs.npmjs.com/cli/v11/commands/npm-run/)：从本技术目录运行已配置脚本，并解析本地工具。